# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os ; import sys
sys.path.insert(0, os.path.abspath(os.path.join('./lib')))

import utilities
import detect_ignition
import harmonics
import psd_waterfall
from psd_waterfall import IgnitionPsdCollector

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 128

import traceback

from scipy.signal import butter, filtfilt, hilbert, welch, csd, get_window

In [ ]:
RANGES  = {'Delta':[1,4],'Theta':[4,8],'Alpha':[8,12],'BetaL':[12,16], 'BetaH':[16,25],'Gamma':[25,45]} ; FS = 128

# Load EEG Data

In [ ]:
FILENAME = "data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv"

ELECTRODES = ['EEG.AF3','EEG.AF4','EEG.F7','EEG.F8','EEG.F3','EEG.F4','EEG.FC5','EEG.FC6','EEG.P7','EEG.P8','EEG.T7','EEG.T8','EEG.O1','EEG.O2']

RECORDS = utilities.load_eeg_csv(FILENAME, electrodes=ELECTRODES)

# Estimate Schumann Harmonics

In [ ]:
HARMONICS = harmonics.estimate_sr_harmonics(RECORDS, sr_channel='EEG.F4', fs=None,
                          f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
                          search_halfband=1.0, nperseg_sec=32.0, overlap=0.5)

SUBHARMONICS = harmonics.estimate_sr_harmonics(RECORDS, sr_channel='EEG.F4', fs=None,
                          f_can=(HARMONICS[0],HARMONICS[0]/2.0,HARMONICS[0]/3.0,HARMONICS[0]/4.0,HARMONICS[0]/5.0, HARMONICS[0]/6.0, HARMONICS[0]/7.0, HARMONICS[0]/8.0),
                          search_halfband=0.1, nperseg_sec=32.0, overlap=0.5)

print(HARMONICS)
print(SUBHARMONICS)

# Detect Ignitions

In [ ]:
# def detect_ignitions_session(
#     RECORDZ: pd.DataFrame,
#     sr_channel: Optional[str] = "EEG.F4",                 # kept for continuity in plots/logs
#     eeg_channels: Optional[List[str]] = None,
#     time_col: str = 'Timestamp',
#     out_dir: str = 'exports_ignitions/S01',
#     # detection params
#     center_hz: float = 7.83, half_bw_hz: float = 0.6,
#     smooth_sec: float = 0.25, z_thresh: float = 2.5,
#     min_isi_sec: float = 2.0, window_sec: float = 20.0, merge_gap_sec: float = 5.0,
#     # validation params (session-level R)
#     R_band: Tuple[float, float] = (8,13), R_win_sec: float = 1.0, R_step_sec: float = 0.25,
#     eta_pre_sec: float = 10.0, eta_post_sec: float = 10.0,
#     # NEW adaptive/event-centric knobs
#     sr_reference: str = 'auto-SSD',       # 'F4' | 'auto-SSD' | 'auto-PLV' | 'auto-PCA'
#     seed_method: str = 'latency',         # 'latency' | 'PGD'
#     pel_band: Tuple[float,float] = (60, 90),
#     electrode_xy: Optional[Dict[str, Tuple[float,float]]] = None,
#     harmonics: Tuple[int,...] = (2,3,4,5,6,7),
#     make_passport: bool = True,
#     show: bool = True,
#     verbose: bool = True
# ) -> Tuple[Dict[str, object], List[Tuple[int,int]]]:

import detect_ignition

out, IGNITION_WINDOWS = detect_ignition.detect_ignitions_session(
    RECORDS, eeg_channels=ELECTRODES,
    center_hz=HARMONICS[0], half_bw_hz=0.35,
    z_thresh=3,
    R_band=(HARMONICS[0]-0.5,HARMONICS[0]+0.5),
    sr_reference='auto-SSD', seed_method='latency',
    pel_band=(35,58), 
    harmonics_hz=HARMONICS,
    eta_pre_sec = 5.0, eta_post_sec = 5.0,
    out_dir='exports_ignitions/S01',
    window_sec=5
)

# Delta 

In [ ]:
# 1) Get crest peaks across events
events_df = out['result']['events'] if 'result' in out else out['events']
DF, _ = detect_ignition.summarize_delta_hotspots(
    RECORDS, events_df,
    eeg_channels=ELECTRODES, #['EEG.F3','EEG.F4','EEG.AF3','EEG.AF4','EEG.F7','EEG.F8','EEG.O1','EEG.O2'],
    combine='mean',
    crest_win=20,
    baseline_offset=(-10, -5),
    f_lo=1.5, f_hi=4,
    top_n=10
)

# 2) Cluster + print
hotspots = detect_ignition.cluster_delta_hotspots_meanshift(DF, z_thresh=2.0, bandwidth_quantile=0.2, fallback_bw=0.05)
pd.options.display.float_format = '{:0.3f}'.format
print("\nDelta surge hotspots (MeanShift, z≥2):")
display(hotspots)


# Animations

In [ ]:
IGN = 5

In [ ]:
anim, path = detect_ignition.animate_delta_psd(
    RECORDS,
    eeg_channels=ELECTRODES, #['EEG.F3','EEG.F4','EEG.AF3','EEG.AF4','EEG.F7','EEG.F8'],
    combine='mean',
    t_range=(IGNITION_WINDOWS[IGN][0],IGNITION_WINDOWS[IGN][1]),
    f_lo=0.5, f_hi=12,
    win_sec=10, step_sec=0.1,
    norm='z', baseline_range=(30,58),
    fps= 48,
    out_path='delta_psd_anim.mp4',
    show_inline=True, fill_alpha=0.15, dyn_ylim=False, ylim_pad=10,            # <-- 10% headroom        # <-- add soft shading
    title='Delta PSD (z-score) — frontal mean'
)

from IPython.display import HTML
HTML(anim.to_jshtml())   # inline without ffmpeg


In [ ]:
# anim, path = detect_ignition.animate_rbp(
#     RECORDS,
#     eeg_channels=ELECTRODES, #['EEG.F3','EEG.F4','EEG.AF3','EEG.AF4','EEG.F7','EEG.F8'],
#     combine='mean',     # or 'mean'/'median'
#     t_range=(IGNITION_WINDOWS[IGN][0],IGNITION_WINDOWS[IGN][1]),
#     win_sec=1.0, step_sec=0.04,
#     fps=24,
#     out_path='rbp_F4_lines_combined.mp4',
#     show_inline=True,
#     view_sec=5.0,         # fixed sliding window
#     fill_alpha=0.15
# )
# import matplotlib as mpl
# mpl.rcParams['animation.embed_limit'] = 64 

# from IPython.display import HTML
# HTML(anim.to_jshtml()) 

In [ ]:
sr_halfband = 0.5
sr_bands = [
    ('SR1', (HARMONICS[0]-sr_halfband,HARMONICS[0]+sr_halfband)), 
    ('2x', (HARMONICS[1]-sr_halfband,HARMONICS[1]+sr_halfband)), 
    ('3x', (HARMONICS[2]-sr_halfband,HARMONICS[2]+sr_halfband)), 
    ('4x', (HARMONICS[3]-sr_halfband,HARMONICS[3]+sr_halfband)),               
    ('5x', (HARMONICS[4]-sr_halfband,HARMONICS[4]+sr_halfband)), 
    ('6x', (HARMONICS[5]-sr_halfband,HARMONICS[5]+sr_halfband)), 
    ('7x', (HARMONICS[6]-sr_halfband,HARMONICS[6]+sr_halfband)), 
    ('8x', (HARMONICS[7]-sr_halfband,HARMONICS[7]+sr_halfband))
]

IGN = 1
anim, path, last_png = detect_ignition.animate_psd_stacked(
    RECORDS,
    eeg_channels=ELECTRODES, #['EEG.F4','EEG.FC6','EEG.P8'],
    combine='mean',
    t_range=(IGNITION_WINDOWS[IGN][0],IGNITION_WINDOWS[IGN][1]),
    # default_bands='delta6',   # 'canonical' | 'schumann' | 'delta6' | 'custom'
    bands=sr_bands,
    win_sec=3, step_sec=0.05,
    fps=48, out_path='psd_stacked_schumann.mp4',
    show_inline=True,
    title='Stacked absolute power — Schumann bands (SR±0.5 Hz) : ', # (SR±0.5 Hz)
    legend_outside=True,
    save_last_frame=True
)
from IPython.display import HTML
HTML(anim.to_jshtml())


# Batch

In [ ]:
# 1) create the collector once
collector = IgnitionPsdCollector(freq_range=(1,45), sort_by=('sr',7.8), normalize=False, sr_tol_hz= .5)

## Epoc X

In [ ]:
import psd_waterfall

# 1) create the collector once
collector = IgnitionPsdCollector(freq_range=(2,30), sort_by=('sr',7.8), normalize=True, sr_tol_hz= .5) # ('sr',7.8)

files = [
    'data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv',
    # 'data/Test_06.11.20_14.28.18.md.pm.bp.csv',
    'data/20201229_29.12.20_11.27.57.md.pm.bp.csv',
    # 'data/med_EPOCX_111270_2021.06.12T09.50.52.04.00.md.bp.csv',
    'data/binaural_EPOCX_111270_2021.06.17T10.04.52.04.00.md.bp.csv',   
    # 'data/hyp_02.01.21_13.51.16.md.pm.bp.csv'
    # 'data/Quality Assessment_MM_EPOCX_111270_2021.02.16T10.51.08.05.00.md.mc.pm.fe.bp.csv'
]



# 2) Output root for all sessions
ROOT_OUT = 'exports_ignitions_batch'
os.makedirs(ROOT_OUT, exist_ok=True)

# 4) Collect per-session summaries
master_rows = []
sc = 1
for fpath in files:

    records = utilities.load_eeg_csv(fpath, electrodes=ELECTRODES)
   
    session_name = os.path.splitext(os.path.basename(fpath))[0]
    out_dir = os.path.join(ROOT_OUT, session_name)
    try:
        print(f"\n=== Processing {session_name} ===")

        harms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.F4', fs=None,
                    f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
                    search_halfband=0.8, nperseg_sec=32.0, overlap=0.5)

        out, ign_windows = detect_ignition.detect_ignitions_session(
            records, eeg_channels=ELECTRODES,
            z_thresh=3,
            # R_band=(harms[0]-0.5,harms[0]+0.5), 
            R_win_sec=3, R_step_sec=0.01,
            # sr_channel="EEG.F4",
            smooth_sec=1, 
            min_isi_sec=2.0, 
            window_sec=5.0, 
            merge_gap_sec=3.0,
            sr_reference='auto-SSD',    # auto-SSD auto-PLV  auto-PCA
            pel_band=(35,58), 
            harmonics_hz=harms,
            harmonic_bw_hz = 0.5, # Optional[float] = None,
            eta_pre_sec = 3.0, eta_post_sec = 3.0,
            out_dir='exports_ignitions_batch/EPOCX/S'+ str(sc),
        )

        fig, freqs, Z, info = psd_waterfall.plot_ignition_psd_waterfall(
            records,elev=10,azim=-80,
            windows=ign_windows,
            fs=128,
            channels=ELECTRODES,
            # band=band,
            # freq_range=fr,
            # average=args.average,
            baseline_windows=None,
            # out_path=args.out,
            title="Session Ignitions: Frequency × Event × 10·log10 PSD",
            average="median"
        )

        # collector.add_precomputed(freqs, Z, session_id=sc, windows=ign_windows, fs=128)
        collector.add_session(
            records, windows=ign_windows, fs=128, session_id=sc,
            band=(1,45), notch=50.0,            # or 50.0
            nperseg_sec=3, overlap=0.75,      # ↑ true spectral resolution = fs/nperseg
            nfft=2048,                          # ↑ grid density (Δf = fs/nfft ~ 0.03125 Hz @ fs=128)
            average='median'
        )

        sc = sc + 1
        
        # Store summary row
        summ = out['summary'].copy()
        summ['session'] = session_name[:30]+" ..."
        summ['n_events'] = summ.get('n_events', 0)
        master_rows.append(summ)

    except Exception as e:
        print(f"[ERROR] {session_name}: {e}")
        traceback.print_exc()
        # add a failed row so you keep the log complete
        master_rows.append({'session': session_name[:20]+" ...", 'n_events': np.nan, 'error': str(e)})

# 5) Save master summary across sessions
master_df = pd.DataFrame(master_rows)
master_csv = os.path.join(ROOT_OUT, 'master_ignition_summary-EPOCX.csv')
master_df.to_csv(master_csv, index=False)

print("\n=== Batch complete ===")
print("Master summary saved to:", master_csv)
print(master_df.fillna('').to_string(index=False))

# 3) one grand visualization at the end
fig, freqs, Z_all, info = collector.plot_grand_waterfall(
    title="All Sessions — Ignition PSD Grand Waterfall",
    view_preset='sr_alignment',     # rotate to make SR alignment obvious
    heatmap_panel=True,             # add a bottom 2D panel for orthographic check
    sr_curtains=True, sr_markers=True, sr_project_base=True, figsize=(20,20),
    elev=10,azim=-90,
)

In [ ]:
# 3) one grand visualization at the end
fig, freqs, Z_all, info = collector.plot_grand_waterfall(
    title="All Sessions — Ignition PSD Grand Waterfall",
    view_preset='sr_alignment',     # rotate to make SR alignment obvious
    heatmap_panel=True,             # add a bottom 2D panel for orthographic check
    sr_curtains=True, sr_markers=True, sr_project_base=True, figsize=(15,20),
    elev=45,azim=-90,
)

## Insight

In [ ]:
INSIGHT_ELECTRODES = ['EEG.AF3','EEG.AF4','EEG.T7','EEG.T8', 'EEG.Pz']

# 1) List your input files (CSV paths)
insight_files = [
    'data/testing_INSIGHT2_111270_2024.02.10T10.25.53.06.00.md.pm.bp.csv',
    'data/shit_INSIGHT2_111270_2024.02.10T13.00.17.06.00.md.pm.bp.csv',
    'data/testing_INSIGHT2_111270_2024.02.12T10.25.21.06.00.md.pm.bp.csv',
    'data/testing_INSIGHT2_111270_2024.02.13T08.26.34.06.00.md.pm.bp.csv',
    # 'data/testing_INSIGHT2_111270_2024.02.15T11.43.15.06.00.md.pm.bp.csv',
    # 'data/test_INSIGHT2_111270_2024.03.01T11.19.02.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.02T10.03.06.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.02T10.17.57.06.00.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.05T07.14.31.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.05T07.02.55.06.00.bp.csv',
    'data/tet_INSIGHT2_111270_2024.03.05T07.49.33.06.00.md.pm.bp.csv',
    'data/test_INSIGHT2_111270_2024.03.08T12.26.06.06.00.md.pm.bp.csv'
]


In [ ]:
# 1) create the collector once
collector = IgnitionPsdCollector(freq_range=(4,30), sort_by=('sr',7.8), normalize=True, sr_tol_hz= .5) # ('sr',7.8)

# 2) Output root for all sessions
ROOT_OUT = 'exports_ignitions_batch'
os.makedirs(ROOT_OUT, exist_ok=True)

# 4) Collect per-session summaries
master_rows = []
sc = 1
for fpath in insight_files:
    records = utilities.load_eeg_csv(fpath, electrodes=INSIGHT_ELECTRODES)
    records = records.iloc[3840:-1920].reset_index(drop=True).copy()
    
    session_name = os.path.splitext(os.path.basename(fpath))[0]
    out_dir = os.path.join(ROOT_OUT, session_name)
    try:
        print(f"\n=== Processing {session_name} ===\n")

        harms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.AF4', fs=None,
                    f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
                    search_halfband=0.8, nperseg_sec=32.0, overlap=0.5)

        subharms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.AF4', fs=None,
                          f_can=(harms[0]/2.0,harms[0]/3.0,harms[0]/4.0,harms[0]/5.0, harms[0]/6.0, harms[0]/7.0, harms[0]/8.0),
                          search_halfband=0.1, nperseg_sec=32.0, overlap=0.5)

        
        
        formatted_list_fstring = [f"{num:.2f}" for num in harms]
        formatted_list_subfstring = [f"{num:.2f}" for num in subharms]
        print(f"Estimate SR harmonics: {formatted_list_fstring}")
       
        

        out, ign_windows = detect_ignition.detect_ignitions_session(
            records, eeg_channels=INSIGHT_ELECTRODES,
            z_thresh=3,
            # R_band=(harms[0]-0.5,harms[0]+0.5), 
            R_win_sec=3, R_step_sec=0.01,
            sr_channel="EEG.AF4",
            smooth_sec=1, 
            min_isi_sec=2.0, 
            window_sec=3.0, 
            merge_gap_sec=3.0,
            sr_reference='auto-SSD',    # auto-SSD auto-PLV  auto-PCA
            pel_band=(35,58), 
            harmonics_hz=harms,
            harmonic_bw_hz = 0.5, # Optional[float] = None,
            eta_pre_sec = 3.0, eta_post_sec = 3.0,
            out_dir='exports_ignitions_batch/INSIGHT/S'+ str(sc),
        )
        #     z_thresh=4,R_band=(harms[0]-0.5,harms[0]+0.5),
        #     sr_channel="EEG.Pz", sr_reference='auto-SSD', seed_method='latency',
        #     pel_band=(35,58),
        #     harmonics_hz=harms,
        #     eta_pre_sec = 5.0, eta_post_sec = 5.0,
        #     out_dir='exports_ignitions_batch/INSIGHT/S'+ str(sc),
        #     window_sec=2
        # )

        fig, freqs, Z, info = psd_waterfall.plot_ignition_psd_waterfall(
            records,elev=10,azim=-80,
            windows=ign_windows,
            fs=128,
            channels=INSIGHT_ELECTRODES,
            # band=band,
            # freq_range=fr,
            # average=args.average,
            baseline_windows=None,
            # out_path=args.out,
            title="Session Ignitions: Frequency × Event × 10·log10 PSD",
            average="mean"
        )

        # collector.add_precomputed(freqs, Z, session_id=sc, windows=ign_windows, fs=128)
        collector.add_session(
            records, windows=ign_windows, fs=128, session_id=sc,
            band=(1,45), notch=50.0,            # or 50.0
            nperseg_sec=1, overlap=0.75,      # ↑ true spectral resolution = fs/nperseg
            nfft=4096,                          # ↑ grid density (Δf = fs/nfft ~ 0.03125 Hz @ fs=128)
            average='median'
        )
        
        # # 1) Get crest peaks across events
        # events_df = out['result']['events'] if 'result' in out else out['events']
        # DF, _ = detect_ignition.summarize_delta_hotspots(
        #     records, events_df,
        #     eeg_channels=['EEG.AF3','EEG.AF4', 'EEG.Pz'],
        #     combine='median',
        #     crest_win=60,
        #     baseline_offset=(-10, -5),
        #     f_lo=1.5, f_hi=4,
        #     top_n=10
        # )

        # print(f"\nEstimate SR subharmonics: {formatted_list_subfstring}")
        
        # # 2) Cluster + print
        # hotspots = detect_ignition.cluster_delta_hotspots_meanshift(DF, z_thresh=2.0, bandwidth_quantile=0.2, fallback_bw=0.05)
        # pd.options.display.float_format = '{:0.3f}'.format
        # print("\nDelta surge hotspots (MeanShift, z≥2):")
        # display(hotspots)



        # ign=1
        # for ign_win in ign_windows:
        #     anim, path, last_png = detect_ignition.animate_psd_stacked(
        #         records,
        #         eeg_channels=INSIGHT_ELECTRODES, #['EEG.F4','EEG.FC6','EEG.P8'],
        #         combine='mean',
        #         t_range=(ign_win[0],ign_win[1]),
        #         default_bands='canonical',
        #         # default_bands='schumann',
        #         # bands=sr_bands,
        #         win_sec=3, step_sec=0.05,
        #         fps=24, out_path="exports_ignitions_batch/INSIGHT/S"+ str(sc)+ "/" + "psd_stacked_canonical_insight_s"+str(sc)+"e"+str(ign)+".mp4",
        #         show_inline=True,
        #         title='Stacked absolute power — canonical bands : ',
        #         legend_outside=True,
        #         save_last_frame=True,
        #         last_frame_path="exports_ignitions_batch/INSIGHT/S"+ str(sc)+ "/" + "psd_stacked_canonical_insight_s"+str(sc)+"e"+str(ign)+".png"
        #     )
        #     ign = ign + 1
        
        # sc = sc+1

        
        # Store summary row
        summ = out['summary'].copy()
        summ['session'] = session_name[:30]+" ..."
        summ['n_events'] = summ.get('n_events', 0)
        master_rows.append(summ)



    except Exception as e:
        print(f"[ERROR] {session_name}: {e}")
        traceback.print_exc()
        # add a failed row so you keep the log complete
        master_rows.append({'session': session_name[:20]+" ...", 'n_events': np.nan, 'error': str(e)})

# 5) Save master summary across sessions
master_df = pd.DataFrame(master_rows)
master_csv = os.path.join(ROOT_OUT, 'master_ignition_summary-INSIGHT.csv')
master_df.to_csv(master_csv, index=False)

print("\n=== Batch complete ===")
print("Master summary saved to:", master_csv)
print(master_df.fillna('').to_string(index=False))

In [ ]:
# 3) one grand visualization at the end
fig, freqs, Z_all, info = collector.plot_grand_waterfall(
    title="All Sessions — Ignition PSD Grand Waterfall",
    view_preset='sr_alignment',     # rotate to make SR alignment obvious
    heatmap_panel=True,             # add a bottom 2D panel for orthographic check
    sr_curtains=True, sr_markers=True, sr_project_base=True, figsize=(15,20),
    elev=45,azim=-90,
)

## Muse

In [ ]:
import utilities

MUSE_ELECTRODES = ['EEG.AF7','EEG.AF8','EEG.TP9','EEG.TP10']

# 1) List your input files (CSV paths)
muse_files = [
    # 'data/test.csv',
    'data/Muse-461E_2019-11-28--16-01-03_1575083390915.csv',
    'data/Muse-461E_2019-11-27--19-26-58_1575321084010.csv',
    'data/Muse-461E_2019-12-03--12-47-46_1575407413023.csv',
    # 'data/Muse-461E_2019-11-20--21-13-30_1574381977511.csv',
    'data/Muse-461E_2019-12-03--20-53-25_1575436553405.csv',
    'data/Muse-461E_2019-12-04--12-05-39_1575491284751.csv',
    'data/Muse-461E_2019-12-05--12-17-34_1575579123713.csv',
    'data/Muse-461E_2019-12-06--14-36-37_1575672417939.csv',
    'data/Muse-461E_2019-12-06--20-54-24_1575694700892.csv',
    'data/Muse-461E_2019-12-06--20-57-59_1575694836983.csv',
    # 'data/Muse-461E_2019-12-06--21-00-26_1575696038038.csv',
    'data/Muse-461E_2019-12-07--13-11-51_1575754159523.csv',
    'data/Muse-461E_2019-12-07--18-28-16_1575772914160.csv',
    'data/Muse-461E_2019-12-15--15-20-11_1576454995317.csv',
    'data/Muse-357D_2019-12-17--14-20-44_1576623304487.csv'
#     'data/Muse-357D_2019-12-24--10-52-16_1577215680302.csv',
#     'data/Muse-461E_2020-07-12--13-30-04_1594576299757.csv'
]

# 2) Output root for all sessions
ROOT_OUT = 'exports_ignitions_batch'
os.makedirs(ROOT_OUT, exist_ok=True)

# 4) Collect per-session summaries
master_rows = []
sc = 1
for fpath in muse_files:
    records = utilities.load_eeg_csv(fpath, electrodes=MUSE_ELECTRODES,device='muse')
    records = records.iloc[7680:-3840].reset_index(drop=True).copy()

    session_name = os.path.splitext(os.path.basename(fpath))[0]
    out_dir = os.path.join(ROOT_OUT, session_name)
    try:
        print(f"\n=== Processing {session_name} ===\n")

        harms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.AF8', fs=None,
                    f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
                    search_halfband=0.8, nperseg_sec=32.0, overlap=0.5)

        formatted_list_fstring = [f"{num:.2f}" for num in harms]
        print(f"Estimate SR harmonics: {formatted_list_fstring}")
        

        out, ign_windows = detect_ignition.detect_ignitions_session(
            records, eeg_channels=MUSE_ELECTRODES,
            z_thresh=4,R_band=(harms[0]-0.4,harms[0]+0.4),
            sr_channel="EEG.AF8", sr_reference='auto-SSD', seed_method='latency',
            pel_band=(35,58),
            harmonics_hz=harms,
            eta_pre_sec = 5.0, eta_post_sec = 5.0,
            out_dir='exports_ignitions_batch/MUSE/S'+ str(sc),window_sec=3
        )

        # ign=1
        # for ign_win in ign_windows:
        #     anim, path, last_png = detect_ignition.animate_psd_stacked(
        #         records,
        #         eeg_channels=MUSE_ELECTRODES, #['EEG.F4','EEG.FC6','EEG.P8'],
        #         combine='mean',
        #         t_range=(ign_win[0],ign_win[1]),
        #         default_bands='canonical',
        #         # default_bands='schumann',
        #         # bands=sr_bands,
        #         win_sec=3, step_sec=0.05,
        #         fps=24, out_path="exports_ignitions_batch/MUSE/S"+ str(sc)+ "/" + "psd_stacked_canonical_muse_s"+str(sc)+"e"+str(ign)+".mp4",
        #         show_inline=True,
        #         title='Stacked absolute power — canonical bands : ',
        #         legend_outside=True,
        #         save_last_frame=True,
        #         last_frame_path="exports_ignitions_batch/MUSE/S"+ str(sc)+ "/" + "psd_stacked_canonical_muse_s"+str(sc)+"e"+str(ign)+".png"
        #     )
        #     ign = ign + 1

        fig, freqs, Z, info = plot_ignition_psd_waterfall(
            records,elev=10,azim=-80,
            windows=ign_windows,
            fs=128,
            channels=MUSE_ELECTRODES,
            # band=band,
            # freq_range=fr,
            # average=args.average,
            baseline_windows=None,
            # out_path=args.out,
            title="Session Ignitions: Frequency × Event × 10·log10 PSD",
            average="mean"
        )

        collector.add_precomputed(freqs, Z, session_id=sc, windows=ign_windows, fs=128)
        
        print("\n")
        sc = sc+1

        
        # Store summary row
        summ = out['summary'].copy()
        summ['session'] = session_name[:30]+" ..."
        summ['n_events'] = summ.get('n_events', 0)
        master_rows.append(summ)



    except Exception as e:
        print(f"[ERROR] {session_name}: {e}")
        traceback.print_exc()
        # add a failed row so you keep the log complete
        master_rows.append({'session': session_name[:20]+" ...", 'n_events': np.nan, 'error': str(e)})

# 5) Save master summary across sessions
master_df = pd.DataFrame(master_rows)
master_csv = os.path.join(ROOT_OUT, 'master_ignition_summary-MUSE.csv')
master_df.to_csv(master_csv, index=False)

print("\n=== Batch complete ===")
print("Master summary saved to:", master_csv)
print(master_df.fillna('').to_string(index=False))

# GRAND WATERFALL

In [ ]:
# 3) one grand visualization at the end
fig, freqs, Z_all, info = collector.plot_grand_waterfall(
    title="All Sessions — Ignition PSD Grand Waterfall",
    view_preset='sr_alignment',     # rotate to make SR alignment obvious
    heatmap_panel=True,             # add a bottom 2D panel for orthographic check
    sr_curtains=True, sr_markers=True, sr_project_base=True, figsize=(20,20),
    elev=10,azim=-90,
)


In [ ]:
files = [
    'data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv',
    'data/Test_06.11.20_14.28.18.md.pm.bp.csv',
    'data/20201229_29.12.20_11.27.57.md.pm.bp.csv',
    # 'data/med_EPOCX_111270_2021.06.12T09.50.52.04.00.md.bp.csv',
    'data/binaural_EPOCX_111270_2021.06.17T10.04.52.04.00.md.bp.csv',   
    # 'data/hyp_02.01.21_13.51.16.md.pm.bp.csv'
    # 'data/Quality Assessment_MM_EPOCX_111270_2021.02.16T10.51.08.05.00.md.mc.pm.fe.bp.csv'
]



# 2) Output root for all sessions
ROOT_OUT = 'exports_ignitions_batch'
os.makedirs(ROOT_OUT, exist_ok=True)

# 4) Collect per-session summaries
master_rows = []
sc = 1


 # sort_by: None | 'session' | 'duration' | 'max' | ('sr', f0)
 #                 ('sr' ,7.83) sorts by power at ~f0.
                     
# 1) create the collector once
collector = IgnitionPsdCollector(freq_range=(4,35), sort_by=('sr',7.83), normalize=True, sr_tol_hz= 0.5)

for fpath in files:

    records = utilities.load_eeg_csv(fpath, electrodes=ELECTRODES)
    
    if fpath == 'data/hyp_02.01.21_13.51.16.md.pm.bp.csv':
        records = records.iloc[7680:-3840].reset_index(drop=True).copy()
    
    session_name = os.path.splitext(os.path.basename(fpath))[0]
    out_dir = os.path.join(ROOT_OUT, session_name)

    harms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.F4', fs=None,
        f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
        search_halfband=0.8, nperseg_sec=32.0, overlap=0.5)

    formatted_list_fstring = [f"{num:.2f}" for num in harms]
    print(f"Estimate SR harmonics: {formatted_list_fstring}")
    
    out, ign_windows = detect_ignition.detect_ignitions_session(
        records, eeg_channels=ELECTRODES,
        z_thresh=3,R_band=(harms[0]-0.4,harms[0]+0.4),
        sr_reference='auto-SSD', seed_method='latency',
        pel_band=(35,58), 
        harmonics_hz=harms,
        eta_pre_sec = 5.0, eta_post_sec = 5.0,
        out_dir='exports_ignitions_batch/EPOCX/S'+ str(sc)
    )
    
    fig, freqs, Z, info = plot_ignition_psd_waterfall(
        records,elev=10,azim=-80,
        windows=ign_windows,
        fs=128,
        channels=ELECTRODES,
        # band=band,
        # freq_range=fr,
        # average=args.average,
        baseline_windows=None,
        # out_path=args.out,
        title="Session Ignitions: Frequency × Event × 10·log10 PSD",
        average="mean"
    )

    collector.add_precomputed(freqs, Z, session_id=sc, windows=ign_windows, fs=128)

    # collector.add_session(
    #     RECORDS=records, windows=ign_windows, fs=128, session_id=sc,
    #     band=(1,45), notch=50.0, nperseg_sec=2.0, overlap=0.5, average='mean',
    #     baseline_windows=None  # or [(0,60)] if you want ΔdB
    # )

    sc = sc + 1

In [ ]:
# fig, freqs, Z, info = plot_session_ignition_psd_sr(
#     RECORDS, windows=IGNITION_WINDOWS, fs=128,
#     # band=(1,45), notch=50.0, freq_range=(1,30), 
#     average='mean',
#     # view/zoom/size
#     figsize=(20, 5),
#     # view_preset='sr_alignment',   # still works; can override with angles below
#     elev=15, azim=-80,            # camera angles
#     dist=7, proj_type='persp',    # zoom & projection
#     # xlim=(6, 10),                 # frequency zoom (Hz)
#     # ylim=(0, len(IGNITION_WINDOWS)),  # event rows
#     # zlim=(-5, 25),                # dB range
#     # SR aids
#     sr_curtains=True, sr_markers=True, sr_project_base=True, heatmap_panel=False,
# )


fig, freqs, Z, info = plot_session_ignition_psd_sr(
    RECORDS, windows=IGNITION_WINDOWS, fs=128,
    # better frequency resolution
    nperseg_sec=3, 
    overlap=.5, 
    nfft=2048,
    average='median',

    # view/zoom/size (camera angles + axis crop)
    figsize=(14, 9),
    view_preset=None,      # don't override your angles
    elev=2, azim=-95,     # -90 gives the SR head-on alignment
    proj_type='persp',     # you can keep this; see note on dist below
    # xlim=(6.8, 10.2),      # <-- reliable zoom in frequency
    # ylim=(0, len(IGNITION_WINDOWS)),   # event rows

    # SR aids
    sr_curtains=True, sr_markers=True, sr_project_base=True,
    heatmap_panel=False,
    # band=(1,11), 
    freq_range=(4,12)

    # vmin=-4, vmax=+6 
)


# EXPERIMENTS

## PSD Waterfall

In [ ]:
"""
Ignition PSD Waterfall — Parameterized API (session + grand)
===========================================================

Drop‑in **replacements** for the existing methods with simple view/size params
baked in. You can now pass these directly:

- `figsize=(W, H)` in inches
- `elev`, `azim` (camera angles, deg)
- `dist` (camera distance; smaller = closer)
- `proj_type` ('persp' or 'ortho')
- `xlim`, `ylim`, `zlim` (axis zooming)

Updated call signatures
-----------------------
1) Per‑session 3D waterfall with SR overlays:
   `plot_session_ignition_psd_sr(..., figsize=None, elev=None, azim=None, dist=None,
                                 proj_type='persp', xlim=None, ylim=None, zlim=None, ...)`

2) Across‑session grand waterfall (collector method):
   `IgnitionPsdCollector.plot_grand_waterfall(..., figsize=None, elev=None, azim=None,
                                              dist=None, proj_type='persp',
                                              xlim=None, ylim=None, zlim=None, ...)`

Paste this cell once (or save as a module) and use your existing code with the
new kwargs.
"""
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Optional, Sequence, Tuple, Dict, Any, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy import signal

SCHUMANN_HZ = [7.83, 14.3, 20.8, 27.3]

# --------------------- helpers ---------------------

def _guess_time_col(df: pd.DataFrame) -> Optional[str]:
    for c in ("time","t","Time","TimeS","seconds","sec","timestamp_s"):
        if c in df.columns: return c
    return None


def _auto_channels(df: pd.DataFrame) -> List[str]:
    eeg = [c for c in df.columns if c.startswith("EEG.")]
    if eeg: return eeg
    time_like = set([_guess_time_col(df), "index","sample","Sample","ms","Millis","timestamp"]) - {None}
    numeric = [c for c in df.columns if c not in time_like and np.issubdtype(df[c].dtype, np.number)]
    chans = [c for c in numeric if c.startswith(("CH","F","P","O","T")) or len(c) <= 5]
    return chans or numeric


def _sec_to_idx(df: pd.DataFrame, s: float, e: float, fs: float, time_col: Optional[str]) -> Tuple[int,int]:
    if time_col is None:
        return max(0,int(np.floor(s*fs))), min(len(df),int(np.ceil(e*fs)))
    t = df[time_col].to_numpy()
    i0 = int(np.searchsorted(t, s, 'left')); i1 = int(np.searchsorted(t, e, 'right'))
    return max(0,i0), min(len(df), i1)


def _bandpass_notch(X: np.ndarray, fs: float, band: Optional[Tuple[float,float]], notch: Optional[float], notch_q=30.0) -> np.ndarray:
    x = X
    if band is not None:
        lo, hi = max(0.01, band[0]), min(0.999*(fs/2.0), band[1])
        sos = signal.iirfilter(4, [lo, hi], rs=40, btype='band', ftype='cheby2', fs=fs, output='sos')
        x = signal.sosfiltfilt(sos, x, axis=0)
    if notch is not None and 0 < notch < fs/2.0:
        b,a = signal.iirnotch(notch, Q=notch_q, fs=fs)
        x = signal.filtfilt(b,a,x, axis=0)
    return x


def _welch_psd(X: np.ndarray, fs: float, nperseg_sec=2.0, overlap=0.5, detrend='linear', nfft=None):
    nper = max(8, int(round(nperseg_sec*fs))); nover = int(round(overlap*nper))
    f, P = signal.welch(X, fs=fs, nperseg=min(nper, X.shape[0]), noverlap=min(nover, X.shape[0]//2),
                        detrend=detrend, axis=0, nfft=nfft, return_onesided=True, scaling='density')
    return f, np.maximum(P, np.finfo(float).eps)


def _aggregate_psd(P: np.ndarray, mode='gfp') -> np.ndarray:
    m = mode.lower()
    if m=='mean': return np.mean(P, axis=1)
    if m=='median': return np.median(P, axis=1)
    if m=='gfp': return np.sqrt(np.mean(P**2, axis=1))
    raise ValueError("average must be 'gfp'|'mean'|'median'")

# --------------------- per‑session compute ---------------------

def compute_psd_by_window_df(df: pd.DataFrame,
                             windows: Sequence[Tuple[float,float]],
                             fs: float,
                             channels: Optional[Sequence[str]] = None,
                             band: Optional[Tuple[float,float]] = None,
                             notch: Optional[float] = None,
                             nperseg_sec: float = 2.0,
                             overlap: float = 0.5,
                             average: str = 'gfp',
                             freq_range: Tuple[float,float] = (1.0, 30.0),
                             baseline_windows: Optional[Sequence[Tuple[float,float]]] = None,
                             detrend: str = 'linear',
                             nfft: Optional[int] = None) -> Tuple[np.ndarray,np.ndarray,Dict[str,Any]]:
    channels = list(channels) if channels is not None else _auto_channels(df)
    if not channels: raise ValueError("No EEG channels found — pass channels=")
    time_col = _guess_time_col(df)

    X_all = df[channels].to_numpy(float)
    X_all = _bandpass_notch(X_all, fs, band, notch)

    baseline_vec = None
    if baseline_windows:
        segs = []
        for (b0,b1) in baseline_windows:
            i0,i1 = _sec_to_idx(df,b0,b1,fs,time_col)
            if i1-i0>8: segs.append(X_all[i0:i1])
        if segs:
            Xb = np.vstack(segs)
            fb,Pb = _welch_psd(Xb,fs,nperseg_sec,overlap,detrend,nfft)
            baseline_vec = _aggregate_psd(Pb,average)

    Z_rows=[]; used=[]; f_ref=None; f_use=None
    fr_lo,fr_hi = freq_range
    for k,(s,e) in enumerate(windows):
        i0,i1 = _sec_to_idx(df,s,e,fs,time_col)
        if i1-i0 < max(8,int(0.5*fs)): continue
        Xi = X_all[i0:i1]
        f,P = _welch_psd(Xi,fs,nperseg_sec,overlap,detrend,nfft)
        if f_ref is None: f_ref=f
        elif len(f)!=len(f_ref) or not np.allclose(f,f_ref):
            pass
        keep=(f>=fr_lo)&(f<=fr_hi); f_use=f[keep]; P_use=P[keep]
        psd_vec = _aggregate_psd(P_use,average)
        if baseline_vec is not None:
            base=baseline_vec[keep]; psd_vec = 10*np.log10(psd_vec/base)
        else:
            psd_vec = 10*np.log10(psd_vec)
        Z_rows.append(psd_vec); used.append(k)

    if not Z_rows: raise RuntimeError("No ignition windows produced valid PSDs.")
    Z = np.vstack(Z_rows)

    meta = dict(
        windows=[windows[i] for i in used], used_indices=used,
        channels=channels, fs=fs, freq_range=freq_range, aggregation=average,
        baseline_windows=list(baseline_windows) if baseline_windows else None,
    )
    return f_use, Z, meta

# --------------------- per‑session plot (with view params) ---------------------

def _resolve_view(view_preset: Optional[str], elev: Optional[float], azim: Optional[float]):
    presets = {
        'isometric': (25, -60),
        'event_headon': (20, 0),
        'freq_headon': (20, -90),
        'topdown': (85, -90),
        'sr_alignment': (20, -90),
    }
    if elev is None or azim is None:
        if view_preset in presets:
            pe, pa = presets[view_preset]
            elev = pe if elev is None else elev
            azim = pa if azim is None else azim
        else:
            de, da = presets['isometric']
            elev = de if elev is None else elev
            azim = da if azim is None else azim
    return elev, azim


def _add_sr_curtains(ax, freqs: np.ndarray, N: int, zmin: float, zmax: float, alpha=0.10, color=(0,0,0)):
    for f0 in SCHUMANN_HZ:
        if freqs[0] <= f0 <= freqs[-1]:
            verts = [[(f0,0,zmin), (f0,N-1,zmin), (f0,N-1,zmax), (f0,0,zmax)]]
            poly = Poly3DCollection(verts, alpha=alpha, facecolor=color, edgecolor='none')
            ax.add_collection3d(poly)
            ax.plot([f0,f0],[0,N-1],[zmax,zmax], color='k', lw=0.8, ls='--', zorder=5)


def _add_sr_markers(ax, freqs: np.ndarray, Z: np.ndarray, cmap_obj, sr_tol_hz=0.35, project_base=True):
    N = Z.shape[0]; y = np.arange(N); zmin = float(np.nanmin(Z))
    for f0 in SCHUMANN_HZ:
        idx = int(np.argmin(np.abs(freqs - f0)))
        if abs(freqs[idx]-f0) > sr_tol_hz: continue
        zvals = Z[:, idx]
        ax.scatter(np.full(N, f0), y, zvals, s=16, c='k', edgecolor='w', linewidth=0.5, depthshade=False, zorder=6)
        if project_base:
            colors = plt.get_cmap('turbo')((zvals - zvals.min()) / max(1e-9, (zvals.max()-zvals.min())))
            ax.scatter(np.full(N, f0), y, np.full(N, zmin), s=12, c=colors, edgecolor='none', depthshade=False, zorder=4)


def plot_waterfall_sr(freqs: np.ndarray, Z: np.ndarray, meta: Dict[str,Any],
                      title: Optional[str] = None,
                      cmap: str = 'turbo',
                      view_preset: Optional[str] = 'sr_alignment',
                      figsize: Optional[Tuple[float,float]] = None,
                      elev: Optional[float] = None, azim: Optional[float] = None,
                      dist: Optional[float] = None, proj_type: str = 'persp',
                      xlim: Optional[Tuple[float,float]] = None,
                      ylim: Optional[Tuple[float,float]] = None,
                      zlim: Optional[Tuple[float,float]] = None,
                      sr_curtains: bool = True, curtain_alpha: float = 0.10,
                      sr_markers: bool = True, sr_tol_hz: float = 0.35,
                      sr_project_base: bool = True,
                      heatmap_panel: bool = True,
                      vmin: Optional[float] = None, vmax: Optional[float] = None) -> Tuple[plt.Figure, Dict]:
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    N, F = Z.shape
    X, Y = np.meshgrid(freqs, np.arange(N))

    if heatmap_panel:
        fig = plt.figure(figsize=figsize or (12, 8.0))
        gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.18)
        ax3d = fig.add_subplot(gs[0,0], projection='3d')
        ax2d = fig.add_subplot(gs[1,0])
    else:
        fig = plt.figure(figsize=figsize or (12, 6.5))
        ax3d = fig.add_subplot(111, projection='3d')
        ax2d = None

    if vmin is None: vmin = np.percentile(Z, 2)
    if vmax is None: vmax = np.percentile(Z, 98)

    cmap_obj = plt.get_cmap(cmap)
    surf = ax3d.plot_surface(X, Y, Z, rstride=1, cstride=1, cmap=cmap_obj,
                             linewidth=0, antialiased=False, shade=True,
                             vmin=vmin, vmax=vmax)

    elev, azim = _resolve_view(view_preset, elev, azim)
    ax3d.view_init(elev=elev, azim=azim)
    # IMPORTANT: set projection first; in 'ortho' mode, dist has no visual effect
    try:
        ax3d.set_proj_type(proj_type)
    except Exception:
        pass
    try:
        if dist is not None:
            ax3d.dist = dist  # has effect only in 'persp' projection on most Matplotlib versions
    except Exception:
        pass

    ax3d.set_xlabel('Frequency (Hz)', labelpad=12)
    ax3d.set_ylabel('Ignition Event', labelpad=10)
    unit = 'ΔdB re: baseline' if meta.get('baseline_windows') else '10·log10 Spectral Density (μV²/Hz)'
    ax3d.set_zlabel(unit, labelpad=10)
    ax3d.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax3d.set_yticks(np.arange(0, N, max(1, N//8)))
    ax3d.set_xlim(freqs[0], freqs[-1])
    if xlim is not None: ax3d.set_xlim(*xlim)
    if ylim is not None: ax3d.set_ylim(*ylim)
    if zlim is not None: ax3d.set_zlim(*zlim)

    zmin, zmax = float(np.nanmin(Z)), float(np.nanmax(Z))

    if sr_curtains:
        _add_sr_curtains(ax3d, freqs, N=N, zmin=zmin, zmax=zmax, alpha=curtain_alpha)
    if sr_markers:
        _add_sr_markers(ax3d, freqs, Z, cmap_obj=cmap_obj, sr_tol_hz=sr_tol_hz, project_base=sr_project_base)

    cbar = fig.colorbar(surf, ax=ax3d, shrink=0.72, aspect=22, pad=0.06)
    cbar.set_label(unit)
    if title: ax3d.set_title(title)

    if ax2d is not None:
        im = ax2d.imshow(Z, aspect='auto', origin='lower', extent=[freqs[0], freqs[-1], 0, N],
                         cmap=cmap_obj, vmin=vmin, vmax=vmax)
        for f0 in SCHUMANN_HZ:
            if freqs[0] <= f0 <= freqs[-1]: ax2d.axvline(f0, ls='--', lw=0.8, color='k', alpha=0.9)
        fig.colorbar(im, ax=ax2d, shrink=0.8, aspect=24, pad=0.02)
        ax2d.set_ylabel('Event'); ax2d.set_xlabel('Frequency (Hz)'); ax2d.set_ylim(0, N)

    fig.tight_layout()

    info = {'N_events': N, 'F_bins': F, 'freqs': freqs,
            'view': {'elev': elev, 'azim': azim, 'dist': dist, 'proj_type': proj_type,
                     'xlim': xlim, 'ylim': ylim, 'zlim': zlim},
            'unit': unit}
    return fig, info


def plot_session_ignition_psd_sr(RECORDS: pd.DataFrame,
                                 windows: Sequence[Tuple[float,float]],
                                 fs: float,
                                 channels: Optional[Sequence[str]] = None,
                                 band: Optional[Tuple[float,float]] = None,
                                 notch: Optional[float] = None,
                                 freq_range: Tuple[float,float] = (1.0, 30.0),
                                 nperseg_sec: float = 2.0,
                                 overlap: float = 0.5,
                                 average: str = 'gfp',
                                 baseline_windows: Optional[Sequence[Tuple[float,float]]] = None,
                                 detrend: str = 'linear',
                                 nfft: Optional[int] = None,
                                 title: Optional[str] = None,
                                 # New view/size params
                                 figsize: Optional[Tuple[float,float]] = None,
                                 view_preset: Optional[str] = 'sr_alignment',
                                 elev: Optional[float] = None, azim: Optional[float] = None,
                                 dist: Optional[float] = None, proj_type: str = 'persp',
                                 xlim: Optional[Tuple[float,float]] = None,
                                 ylim: Optional[Tuple[float,float]] = None,
                                 zlim: Optional[Tuple[float,float]] = None,
                                 # SR aids
                                 sr_curtains: bool = True, curtain_alpha: float = 0.10,
                                 sr_markers: bool = True, sr_tol_hz: float = 0.35,
                                 sr_project_base: bool = True,
                                 heatmap_panel: bool = True,
                                 vmin: Optional[float] = None, vmax: Optional[float] = None,
                                 ) -> Tuple[plt.Figure, np.ndarray, np.ndarray, Dict]:
    freqs, Z, meta = compute_psd_by_window_df(
        df=RECORDS, windows=windows, fs=fs, channels=channels,
        band=band, notch=notch, nperseg_sec=nperseg_sec, overlap=overlap,
        average=average, freq_range=freq_range,
        baseline_windows=baseline_windows, detrend=detrend, nfft=nfft,
    )
    fig, info = plot_waterfall_sr(
        freqs, Z, meta, title=title, cmap='turbo',
        view_preset=view_preset, figsize=figsize, elev=elev, azim=azim,
        dist=dist, proj_type=proj_type, xlim=xlim, ylim=ylim, zlim=zlim,
        sr_curtains=sr_curtains, curtain_alpha=curtain_alpha,
        sr_markers=sr_markers, sr_tol_hz=sr_tol_hz, sr_project_base=sr_project_base,
        heatmap_panel=heatmap_panel, vmin=vmin, vmax=vmax,
    )
    return fig, freqs, Z, info

# --------------------- across‑session collector ---------------------

@dataclass
class RowMeta:
    session_id: str
    event_index: int
    start_sec: float
    end_sec: float
    duration: float
    fs: float


class IgnitionPsdCollector:
    def __init__(self,
                 freq_range: Tuple[float,float] = (1.0, 30.0),
                 sort_by: Optional[Union[str,Tuple[str,float]]] = None,
                 normalize: bool = False,
                 sr_freqs: Sequence[float] = tuple(SCHUMANN_HZ),
                 sr_tol_hz: float = 0.35):
        self.freq_range = freq_range
        self.sort_by = sort_by
        self.normalize = normalize
        self.sr_freqs = np.array(sr_freqs, float)
        self.sr_tol_hz = float(sr_tol_hz)
        self._freqs: Optional[np.ndarray] = None
        self._Z_rows: List[np.ndarray] = []
        self._rows_meta: List[RowMeta] = []
        self._session_breaks: List[int] = []
        self._last_session_id: Optional[str] = None

    def add_session(self,
                    RECORDS: pd.DataFrame,
                    windows: Sequence[Tuple[float,float]],
                    fs: float,
                    session_id: str,
                    channels: Optional[Sequence[str]] = None,
                    band: Optional[Tuple[float,float]] = None,
                    notch: Optional[float] = None,
                    nperseg_sec: float = 2.0,
                    overlap: float = 0.5,
                    average: str = 'gfp',
                    baseline_windows: Optional[Sequence[Tuple[float,float]]] = None,
                    detrend: str = 'linear',
                    nfft: Optional[int] = None) -> Tuple[np.ndarray,np.ndarray,Dict[str,Any]]:
        freqs, Z, meta = compute_psd_by_window_df(
            RECORDS, windows, fs, channels, band, notch,
            nperseg_sec, overlap, average, self.freq_range,
            baseline_windows, detrend, nfft
        )
        self.add_precomputed(freqs, Z, session_id=session_id, windows=meta['windows'], fs=fs)
        return freqs, Z, meta

    def add_precomputed(self,
                        freqs: np.ndarray,
                        Z: np.ndarray,
                        session_id: str,
                        windows: Optional[Sequence[Tuple[float,float]]] = None,
                        fs: Optional[float] = None) -> None:
        freqs = np.asarray(freqs, float)
        Z = np.asarray(Z, float)
        if Z.ndim != 2: raise ValueError("Z must be 2D (events × freqs)")
        if self._freqs is None:
            self._freqs = freqs.copy()
        else:
            f_lo = max(self._freqs[0], freqs[0]); f_hi = min(self._freqs[-1], freqs[-1])
            if f_hi <= f_lo:
                raise RuntimeError("No overlapping frequency range across sessions")
            keep_master = (self._freqs >= f_lo) & (self._freqs <= f_hi)
            if not np.all(keep_master):
                self._freqs = self._freqs[keep_master]
                self._Z_rows = [row[keep_master] for row in self._Z_rows]
            if not np.allclose(freqs, self._freqs):
                Z = np.vstack([np.interp(self._freqs, freqs, row, left=np.nan, right=np.nan) for row in Z])
                good = ~np.any(np.isnan(Z), axis=0)
                if not np.all(good):
                    self._freqs = self._freqs[good]
                    self._Z_rows = [row[good] for row in self._Z_rows]
                    Z = Z[:, good]
        if self.normalize:
            Z = Z - Z.mean(axis=1, keepdims=True)
        base_count = len(self._Z_rows)
        for i in range(Z.shape[0]):
            self._Z_rows.append(Z[i])
            s,e = (windows[i] if windows is not None else (np.nan, np.nan))
            self._rows_meta.append(RowMeta(
                session_id=session_id,
                event_index=i,
                start_sec=float(s), end_sec=float(e), duration=float(e)-float(s) if (np.isfinite(s) and np.isfinite(e)) else np.nan,
                fs=float(fs) if fs is not None else np.nan,
            ))
        if self._last_session_id != session_id:
            self._session_breaks.append(base_count)
            self._last_session_id = session_id

    @property
    def freqs(self) -> np.ndarray:
        if self._freqs is None: raise RuntimeError("Collector is empty")
        return self._freqs

    def _row_df(self) -> pd.DataFrame:
        d = {
            'session_id': [m.session_id for m in self._rows_meta],
            'event_index': [m.event_index for m in self._rows_meta],
            'start_sec': [m.start_sec for m in self._rows_meta],
            'end_sec': [m.end_sec for m in self._rows_meta],
            'duration': [m.duration for m in self._rows_meta],
            'fs': [m.fs for m in self._rows_meta],
        }
        return pd.DataFrame(d)

    def to_dataframe(self) -> pd.DataFrame:
        df_meta = self._row_df(); F = len(self.freqs); rows = []
        for rid, row in enumerate(self._Z_rows):
            rows.append(pd.DataFrame({
                'row_id': rid, 'freq': self.freqs, 'value': row,
                **{k: df_meta.iloc[rid][k] for k in df_meta.columns}
            }))
        return pd.concat(rows, ignore_index=True)

    def _sort_indices(self) -> np.ndarray:
        n = len(self._Z_rows)
        if n == 0: return np.array([], int)
        if self.sort_by is None:
            return np.arange(n)
        key = self.sort_by
        if key == 'session':
            df = self._row_df();
            return np.lexsort((df['event_index'].to_numpy(), df['session_id'].to_numpy()))
        if key == 'duration':
            df = self._row_df(); return np.argsort(df['duration'].to_numpy())
        if key == 'max':
            vmax = np.array([row.max() for row in self._Z_rows]); return np.argsort(-vmax)
        if isinstance(key, tuple) and key[0] == 'sr':
            f0 = float(key[1]); idx = int(np.argmin(np.abs(self.freqs - f0)))
            vals = np.array([row[idx] for row in self._Z_rows]); return np.argsort(-vals)
        return np.arange(n)

    def plot_heatmap(self, title: Optional[str] = None, cmap: str = 'turbo', annotate: bool = True,
                     vmin: Optional[float] = None, vmax: Optional[float] = None,
                     sr_markers: bool = True) -> Tuple[plt.Figure, Dict[str,Any]]:
        if self._freqs is None or not self._Z_rows:
            raise RuntimeError("Collector is empty")
        order = self._sort_indices(); Z_sorted = np.vstack([self._Z_rows[i] for i in order])
        if vmin is None: vmin = np.percentile(Z_sorted, 2)
        if vmax is None: vmax = np.percentile(Z_sorted, 98)

        fig, ax = plt.subplots(figsize=(12, 8))
        im = ax.imshow(Z_sorted, aspect='auto', origin='lower',
                       extent=[self.freqs[0], self.freqs[-1], 0, Z_sorted.shape[0]],
                       cmap=cmap, vmin=vmin, vmax=vmax)
        for f0 in SCHUMANN_HZ:
            if self.freqs[0] <= f0 <= self.freqs[-1]: ax.axvline(f0, ls='--', lw=0.8, color='k', alpha=0.9)
        fig.colorbar(im, ax=ax, shrink=0.8, aspect=24, pad=0.02).set_label('ΔdB or 10·log10 μV²/Hz')
        ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Ignition Event (all sessions)')
        if title: ax.set_title(title)
        if annotate:
            df = self._row_df().iloc[order]
            step = max(1, Z_sorted.shape[0] // 12)
            yticks = list(range(0, Z_sorted.shape[0], step))
            labels = [f"{df.iloc[i].session_id}:{int(df.iloc[i].event_index)}" for i in yticks]
            ax.set_yticks(np.array(yticks)+0.5); ax.set_yticklabels(labels)
        fig.tight_layout()
        info = {'N_events': Z_sorted.shape[0], 'F_bins': Z_sorted.shape[1], 'freqs': self.freqs}
        return fig, info

    def plot_grand_waterfall(self,
                             title: Optional[str] = None,
                             cmap: str = 'turbo',
                             view_preset: str = 'sr_alignment',
                             heatmap_panel: bool = True,
                             sr_curtains: bool = True,
                             sr_markers: bool = True,
                             sr_project_base: bool = True,
                             vmin: Optional[float] = None,
                             vmax: Optional[float] = None,
                             # New view/size params
                             figsize: Optional[Tuple[float,float]] = None,
                             elev: Optional[float] = None,
                             azim: Optional[float] = None,
                             dist: Optional[float] = None,
                             proj_type: str = 'persp',
                             xlim: Optional[Tuple[float,float]] = None,
                             ylim: Optional[Tuple[float,float]] = None,
                             zlim: Optional[Tuple[float,float]] = None,
                             ) -> Tuple[plt.Figure, np.ndarray, np.ndarray, Dict[str,Any]]:
        if self._freqs is None or not self._Z_rows:
            raise RuntimeError("Collector is empty")
        order = self._sort_indices(); Z_sorted = np.vstack([self._Z_rows[i] for i in order])
        fig, info = _plot_waterfall_any(
            freqs=self.freqs, Z=Z_sorted, title=title, cmap=cmap,
            view_preset=view_preset, heatmap_panel=heatmap_panel,
            sr_curtains=sr_curtains, sr_markers=sr_markers, sr_project_base=sr_project_base,
            vmin=vmin, vmax=vmax,
            figsize=figsize, elev=elev, azim=azim, dist=dist, proj_type=proj_type,
            xlim=xlim, ylim=ylim, zlim=zlim,
        )
        return fig, self.freqs, Z_sorted, info

# --------------------- shared 3D plot (grand) ---------------------

def _plot_waterfall_any(freqs: np.ndarray, Z: np.ndarray, title: Optional[str] = None, cmap: str = 'turbo',
                        view_preset: str = 'sr_alignment', heatmap_panel: bool = True,
                        sr_curtains: bool = True, sr_markers: bool = True, sr_project_base: bool = True,
                        vmin: Optional[float] = None, vmax: Optional[float] = None,
                        # New view/size params
                        figsize: Optional[Tuple[float,float]] = None,
                        elev: Optional[float] = None, azim: Optional[float] = None,
                        dist: Optional[float] = None, proj_type: str = 'persp',
                        xlim: Optional[Tuple[float,float]] = None,
                        ylim: Optional[Tuple[float,float]] = None,
                        zlim: Optional[Tuple[float,float]] = None,
                        ) -> Tuple[plt.Figure, Dict[str,Any]]:
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    N, F = Z.shape
    X, Y = np.meshgrid(freqs, np.arange(N))

    if heatmap_panel:
        fig = plt.figure(figsize=figsize or (12, 8.5))
        gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.18)
        ax3d = fig.add_subplot(gs[0,0], projection='3d')
        ax2d = fig.add_subplot(gs[1,0])
    else:
        fig = plt.figure(figsize=figsize or (12, 7.0))
        ax3d = fig.add_subplot(111, projection='3d')
        ax2d = None

    if vmin is None: vmin = np.percentile(Z, 2)
    if vmax is None: vmax = np.percentile(Z, 98)

    cmap_obj = plt.get_cmap(cmap)
    surf = ax3d.plot_surface(X, Y, Z, rstride=1, cstride=1, cmap=cmap_obj,
                             linewidth=0, antialiased=False, shade=True,
                             vmin=vmin, vmax=vmax)

    e, a = _resolve_view(view_preset, elev, azim)
    ax3d.view_init(elev=e, azim=a)
    # set projection first; 'dist' only affects 'persp'
    try:
        ax3d.set_proj_type(proj_type)
    except Exception:
        pass
    try:
        if dist is not None:
            ax3d.dist = dist
    except Exception:
        pass

    ax3d.set_xlabel('Frequency (Hz)', labelpad=12)
    ax3d.set_ylabel('Ignition Event (all sessions)', labelpad=10)
    ax3d.set_zlabel('ΔdB or 10·log10 μV²/Hz', labelpad=10)
    ax3d.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax3d.set_yticks(np.arange(0, N, max(1, N//10)))
    ax3d.set_xlim(freqs[0], freqs[-1])
    if xlim is not None: ax3d.set_xlim(*xlim)
    if ylim is not None: ax3d.set_ylim(*ylim)
    if zlim is not None: ax3d.set_zlim(*zlim)

    zmin, zmax = float(np.nanmin(Z)), float(np.nanmax(Z))

    if sr_curtains:
        _add_sr_curtains(ax3d, freqs, N=N, zmin=zmin, zmax=zmax, alpha=0.10)
    if sr_markers:
        # reuse markers helper from session plot
        _add_sr_markers(ax3d, freqs, Z, cmap_obj=cmap_obj, sr_tol_hz=0.35, project_base=sr_project_base)

    cbar = fig.colorbar(surf, ax=ax3d, shrink=0.72, aspect=22, pad=0.06)
    cbar.set_label('ΔdB or 10·log10 μV²/Hz')
    if title: ax3d.set_title(title)

    if ax2d is not None:
        im = ax2d.imshow(Z, aspect='auto', origin='lower', extent=[freqs[0], freqs[-1], 0, N],
                         cmap=cmap_obj, vmin=vmin, vmax=vmax)
        for f0 in SCHUMANN_HZ:
            if freqs[0] <= f0 <= freqs[-1]: ax2d.axvline(f0, ls='--', lw=0.8, color='k', alpha=0.9)
        fig.colorbar(im, ax=ax2d, shrink=0.8, aspect=24, pad=0.02)
        ax2d.set_ylabel('Event'); ax2d.set_xlabel('Frequency (Hz)'); ax2d.set_ylim(0, N)

    fig.tight_layout()

    info = {'N_events': N, 'F_bins': F, 'freqs': freqs,
            'view': {'elev': e, 'azim': a, 'dist': dist, 'proj_type': proj_type,
                     'xlim': xlim, 'ylim': ylim, 'zlim': zlim}}
    return fig, info


# --------------------- legacy API shim: plot_ignition_psd_waterfall ---------------------

def plot_ignition_psd_waterfall(csv_or_df: Union[str, pd.DataFrame],
                                windows: Sequence[Tuple[float, float]],
                                fs: float,
                                channels: Optional[Sequence[str]] = None,
                                band: Optional[Tuple[float, float]] = None,
                                notch: Optional[float] = None,
                                freq_range: Tuple[float, float] = (1.0, 30.0),
                                nperseg_sec: float = 2.0,
                                overlap: float = 0.5,
                                average: str = 'gfp',
                                baseline_windows: Optional[Sequence[Tuple[float, float]]] = None,
                                detrend: str = 'linear',
                                nfft: Optional[int] = None,
                                title: Optional[str] = None,
                                out_path: Optional[Union[str, 'Path']] = None,
                                # New view/size params (same as SR variant)
                                figsize: Optional[Tuple[float, float]] = None,
                                view_preset: Optional[str] = 'isometric',
                                elev: Optional[float] = None,
                                azim: Optional[float] = None,
                                dist: Optional[float] = None,
                                proj_type: str = 'persp',
                                xlim: Optional[Tuple[float, float]] = None,
                                ylim: Optional[Tuple[float, float]] = None,
                                zlim: Optional[Tuple[float, float]] = None,
                                # Plot cosmetics
                                cmap: str = 'turbo',
                                heatmap_panel: bool = False,
                                vmin: Optional[float] = None,
                                vmax: Optional[float] = None,
                                ) -> Tuple[plt.Figure, np.ndarray, np.ndarray, Dict]:
    """Backward‑compatible, but now with **figsize / camera / zoom** parameters.

    Accepts either a DataFrame or a CSV path, computes PSD per ignition window,
    and renders a 3D waterfall. By default this variant **does not** draw SR
    curtains/markers (use `plot_session_ignition_psd_sr` if you want them), but
    exposes the same view/zoom controls.
    """
    import pathlib

    if isinstance(csv_or_df, pd.DataFrame):
        df = csv_or_df
    elif isinstance(csv_or_df, (str, pathlib.Path)):
        df = pd.read_csv(csv_or_df)
    else:
        raise TypeError("csv_or_df must be a DataFrame or CSV filepath")

    freqs, Z, meta = compute_psd_by_window_df(
        df=df,
        windows=windows,
        fs=fs,
        channels=channels,
        band=band,
        notch=notch,
        nperseg_sec=nperseg_sec,
        overlap=overlap,
        average=average,
        freq_range=freq_range,
        baseline_windows=baseline_windows,
        detrend=detrend,
        nfft=nfft,
    )

    # Reuse the SR plotter but keep SR overlays off for this legacy function
    fig, info = plot_waterfall_sr(
        freqs=freqs,
        Z=Z,
        meta=meta,
        title=title,
        cmap=cmap,
        view_preset=view_preset,
        figsize=figsize,
        elev=elev,
        azim=azim,
        dist=dist,
        proj_type=proj_type,
        xlim=xlim,
        ylim=ylim,
        zlim=zlim,
        sr_curtains=False,
        sr_markers=False,
        sr_project_base=False,
        heatmap_panel=heatmap_panel,
        vmin=vmin,
        vmax=vmax,
    )

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches='tight')

    return fig, freqs, Z, info


In [ ]:
fig, freqs, Z, info = plot_ignition_psd_waterfall(
    RECORDS,
    elev=5,azim=-80,
    windows=IGNITION_WINDOWS,fs=128,
    channels=ELECTRODES,
    # band=band,
    # freq_range=fr,
    # baseline_windows=[(30,90)],
    # out_path=args.out,
    title="Session Ignitions: Frequency × Event × 10·log10 PSD",
    average="mean",
    proj_type='persp', 
    dist=0.05,
    # nperseg_sec=2, 
    # overlap=0.5, 
    nfft=2048
)

In [ ]:
fig, freqs, Z, info = plot_ignition_psd_waterfall(
    RECORDS, windows=IGNITION_WINDOWS, fs=128,
    band=(1,45), notch=50.0, freq_range=(3,30),
    nperseg_sec=1.0, overlap=0.75, nfft=2048,
    title="Session Ignitions — Plain Waterfall",
    figsize=(14,9),
    view_preset='sr_alignment',
    proj_type='persp',     # <-- perspective so dist works
    dist=1,                # smaller = closer
    elev=12, azim=-70,
    # xlim=(6.5, 10.5),
    heatmap_panel=False
)


In [ ]:
files = [
    'data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv',
    'data/Test_06.11.20_14.28.18.md.pm.bp.csv',
    'data/20201229_29.12.20_11.27.57.md.pm.bp.csv',
    'data/med_EPOCX_111270_2021.06.12T09.50.52.04.00.md.bp.csv',
    'data/binaural_EPOCX_111270_2021.06.17T10.04.52.04.00.md.bp.csv',   
    'data/hyp_02.01.21_13.51.16.md.pm.bp.csv'
    # 'data/Quality Assessment_MM_EPOCX_111270_2021.02.16T10.51.08.05.00.md.mc.pm.fe.bp.csv'
]



# 2) Output root for all sessions
ROOT_OUT = 'exports_ignitions_batch'
os.makedirs(ROOT_OUT, exist_ok=True)

# 4) Collect per-session summaries
master_rows = []
sc = 1
for fpath in files:

    records = utilities.load_eeg_csv(fpath, electrodes=ELECTRODES)
    
    if fpath == 'data/hyp_02.01.21_13.51.16.md.pm.bp.csv':
        records = records.iloc[7680:-3840].reset_index(drop=True).copy()
    
    session_name = os.path.splitext(os.path.basename(fpath))[0]
    out_dir = os.path.join(ROOT_OUT, session_name)

    harms = harmonics.estimate_sr_harmonics(records, sr_channel='EEG.F4', fs=None,
        f_can=(7.83, 14.3, 20.8, 27.3, 33.8, 40.3, 46.8, 53.3),
        search_halfband=0.8, nperseg_sec=32.0, overlap=0.5)

    formatted_list_fstring = [f"{num:.2f}" for num in harms]
    print(f"Estimate SR harmonics: {formatted_list_fstring}")
    
    out, ign_windows = detect_ignition.detect_ignitions_session(
        records, eeg_channels=ELECTRODES,
        z_thresh=3,R_band=(harms[0]-0.4,harms[0]+0.4),
        sr_reference='auto-SSD', seed_method='latency',
        pel_band=(35,58), 
        harmonics_hz=harms,
        eta_pre_sec = 5.0, eta_post_sec = 5.0,
        out_dir='exports_ignitions_batch/EPOCX/S'+ str(sc)
    )
    
    fig, freqs, Z, info = plot_ignition_psd_waterfall(
        records,elev=10,azim=-80,
        windows=ign_windows,
        fs=128,
        channels=ELECTRODES,
        # band=band,
        # freq_range=fr,
        # average=args.average,
        baseline_windows=None,
        # out_path=args.out,
        title="Session Ignitions: Frequency × Event × 10·log10 PSD",
        average="mean"
    )